In [218]:
import time
from enum import Enum
from dataclasses import dataclass
from typing import Tuple

import numpy as np
from termcolor import colored, cprint

from utils import Converts

from XPlaneConnectX import XPlaneConnectX
from ics_connector import ICSInputs, ICSOutputs, ICSBenchConnector

In [219]:
NAN = float('nan')

# UUEE 06R
RWY_START_LAT = 55.967296600
RWY_START_LON = 37.387516022
RWY_END_LAT = 55.975555420
RWY_END_LON = 37.442829132
RWY_HEADING_TRUE = 75  # Истинный курс полосы для инициализации
ELEVATION_MSL = 619.0 * Converts.FT_TO_M  # Высота порога над уровнем моря (620 фута в метрах)
ELEVATION_AIRCRAFT = 3 * Converts.FT_TO_M  # Высота ЛА над ВПП

DT = 0.05
FREQ = int(1.0 / DT)

INITIAL_SPEED_KTS = 200.0
TARGET_SPEED_KTS = 10.0

MC21_SETUP = dict(
    speed_knots=INITIAL_SPEED_KTS,
    descent_rate_fpm=200.0,
    pitch_deg=0.0
)

xpc = XPlaneConnectX(ip="127.0.0.1", port=49000)

In [220]:
def setup_touchdown_uuee(xpc: XPlaneConnectX, speed_knots: float = 140.0, descent_rate_fpm: float = 120.0,
                         pitch_deg: float = 0.0):
    """
    Мгновенно воссоздает этап касания (touchdown) самолета на ВПП 24C аэропорта Шереметьево (UUEE)
    с последующим корректным физическим расчетом пробега, обжатия шасси и торможения.

    :param xpc: Экземпляр инициализированного класса XPlaneConnectX
    :param speed_knots: Путевая скорость самолета в узлах в момент касания
    :param descent_rate_fpm: Вертикальная скорость снижения в футах в минуту (положительное число)
    :param pitch_deg: Угол тангажа (типичный угол кабрирования при касании основных стоек)
    """
    # 1. Замораживаем физическое время симулятора.
    # Это необходимо, чтобы все сетевые пакеты (позиция, конфигурация, скорости)
    # применились в рамках одного расчетного кадра до начала обсчета физики.
    xpc.pauseSIM(True)
    time.sleep(0.1)  # Короткая пауза для стабилизации сетевого цикла X-Plane

    roll_deg = 0.0  # Крен равен нулю (крылья параллельны ВПП)

    # 3. Установка пространственного положения планера
    xpc.sendPOSI(lat=RWY_START_LAT, lon=RWY_START_LON, elev=ELEVATION_MSL + ELEVATION_AIRCRAFT, phi=roll_deg,
                 theta=pitch_deg,
                 psi_true=RWY_HEADING_TRUE)

    # 4. Конфигурация органов управления и механизации крыла
    # Тяга на малый газ (0.0), шасси выпущено (1), закрылки на максимум (1.0), спидбрейки армированы (-0.5)
    xpc.sendCTRL(lat_control=0.0, lon_control=0.0, rudder_control=0.0,
                 throttle=0.0, gear=1, flaps=1.0, speedbrakes=-0.5, park_brake=0.0)

    # 5. Расчет проекций векторов скоростей в локальной системе координат X-Plane (OpenGL)
    # Перевод исходных параметров в систему СИ (метры в секунду)
    v_ground_ms = speed_knots * Converts.KTS_TO_MS
    v_vertical_ms = -abs(descent_rate_fpm) * Converts.FTM_TO_MS  # Гарантируем отрицательное значение (движение вниз)

    # Перевод истинного курса в радианы
    heading_rad = np.radians(RWY_HEADING_TRUE)

    # Математический расчет проекций скоростей на оси декартовой сетки X-Plane:
    # Ось X направлена на Восток, ось Z — на Юг.
    local_vx = v_ground_ms * np.sin(heading_rad)
    local_vz = -v_ground_ms * np.cos(heading_rad)
    local_vy = v_vertical_ms

    # 6. Запись векторов скоростей напрямую в DataRefs через команду DREF
    xpc.sendDREF("sim/flightmodel/position/local_vx", local_vx)
    xpc.sendDREF("sim/flightmodel/position/local_vy", local_vy)
    xpc.sendDREF("sim/flightmodel/position/local_vz", local_vz)

    # Демпфирование (обнуление) угловых скоростей для предотвращения паразитного вращения планера
    xpc.sendDREF("sim/flightmodel/position/P", 0.0)
    xpc.sendDREF("sim/flightmodel/position/Q", 0.0)
    xpc.sendDREF("sim/flightmodel/position/R", 0.0)

    time.sleep(0.1)

In [221]:
class FailureMode(Enum):
    NONE = 0

    GEAR_CONFIG = 1

    ENGINE_OUT_LEFT = 2
    ENGINE_OUT_RIGHT = 3

    NWS_FAIL = 4

    REVERSE_LEFT_FAIL = 5
    REVERSE_RIGHT_FAIL = 6

    THRUST_LEFT_DEGRADED = 7
    THRUST_RIGHT_DEGRADED = 8


@dataclass
class FailureState:
    steering_eff = 1.0

    brake_left_eff = 1.0
    brake_right_eff = 1.0

    reverse_left_eff = 1.0
    reverse_right_eff = 1.0

    thrust_left_eff = 1.0
    thrust_right_eff = 1.0

    gear_conflict = False


class FailureManager:
    def __init__(self):
        self.state = FailureState()

    def reset(self):
        self.state = FailureState()

    def activate(self, mode):
        match mode:
            case FailureMode.NWS_FAIL:
                self.state.steering_eff = 0.0

            case FailureMode.ENGINE_OUT_LEFT:
                self.state.thrust_left_eff = 0.0

            case FailureMode.ENGINE_OUT_RIGHT:
                self.state.thrust_right_eff = 0.0

            case FailureMode.REVERSE_LEFT_FAIL:
                self.state.reverse_left_eff = 0.0

            case FailureMode.REVERSE_RIGHT_FAIL:
                self.state.reverse_right_eff = 0.0

            case FailureMode.GEAR_CONFIG:
                self.state.gear_conflict = True
                self.state.brake_left_eff = 0.6
                self.state.brake_right_eff = 0.6

In [222]:
# TODO
# class Telemetry:
#     def __init__(self, xpc: XPlaneConnectX):
#         self.xpc = xpc


class RunwayTracker:
    def __init__(self, lookahead_min=15.0, lookahead_gain=1.5, xte_gain=1.0):
        self.R = 6371008.7714  # Средний радиус Земли в метрах

        self.theta_rwy = self.bearing(RWY_START_LAT, RWY_START_LON, RWY_END_LAT, RWY_END_LON)

        self.lookahead_min = lookahead_min
        self.lookahead_gain = lookahead_gain
        self.xte_gain = xte_gain

        self.rwy_heading = np.radians(RWY_HEADING_TRUE)

    @staticmethod
    def wrap_pi(angle):
        return (angle + np.pi) % (2 * np.pi) - np.pi

    @staticmethod
    def wrap_deg(angle):
        return (angle + 180.0) % 360.0 - 180.0

    @staticmethod
    def bearing(lat1: float, lon1: float, lat2: float, lon2: float):
        phi1 = np.radians(lat1)
        phi2 = np.radians(lat2)

        dlon = np.radians(lon2 - lon1)

        y = np.sin(dlon) * np.cos(phi2)
        x = (np.cos(phi1) * np.sin(phi2) - np.sin(phi1) * np.cos(phi2) * np.cos(dlon))

        return np.arctan2(y, x)

    def haversine_distance(self, lat1: float, lon1: float, lat2: float, lon2: float):
        phi1 = np.radians(lat1)
        phi2 = np.radians(lat2)

        dphi = phi2 - phi1
        dlon = np.radians(lon2 - lon1)

        a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlon / 2.0) ** 2

        return self.R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    def destination(self, lat, lon, bearing, distance):
        """
        Решение прямой геодезической задачи.
        """

        phi1 = np.radians(lat)
        lam1 = np.radians(lon)

        delta = distance / self.R

        phi2 = np.arcsin(
            np.sin(phi1) * np.cos(delta)
            + np.cos(phi1) * np.sin(delta) * np.cos(bearing)
        )

        lam2 = lam1 + np.arctan2(
            np.sin(bearing) * np.sin(delta) * np.cos(phi1),
            np.cos(delta) - np.sin(phi1) * np.sin(phi2)
        )

        return np.degrees(phi2), np.degrees(lam2)

    def guidance(self,
                 aircraft_lat,
                 aircraft_lon,
                 aircraft_heading_deg,
                 ground_speed):
        d13 = self.haversine_distance(
            RWY_START_LAT,
            RWY_START_LON,
            aircraft_lat,
            aircraft_lon
        ) / self.R

        theta13 = self.bearing(
            RWY_START_LAT,
            RWY_START_LON,
            aircraft_lat,
            aircraft_lon
        )

        theta12 = self.rwy_heading

        # Cross-track
        xte = np.arcsin(
            np.sin(d13) *
            np.sin(theta13 - theta12)
        ) * self.R

        # Along-track
        along = np.arctan2(
            np.sin(d13) * np.cos(theta13 - theta12),
            np.cos(d13)
        ) * self.R

        # LookAhead
        lookahead = self.lookahead_min + self.lookahead_gain * ground_speed

        target_distance = along + lookahead

        target_lat, target_lon = self.destination(
            RWY_START_LAT,
            RWY_START_LON,
            theta12,
            target_distance
        )

        desired_heading = self.bearing(
            aircraft_lat,
            aircraft_lon,
            target_lat,
            target_lon
        )

        aircraft_heading = np.radians(aircraft_heading_deg)

        heading_error = self.wrap_pi(
            desired_heading - aircraft_heading
        )

        # Stanley-подобная коррекция
        heading_error += np.arctan2(
            -self.xte_gain * xte,
            max(lookahead, 1.0)
        )

        heading_error = self.wrap_pi(heading_error)

        return {
            "xte": xte,
            "along": along,
            "lookahead": lookahead,
            "heading_error_deg": np.degrees(heading_error),
            "desired_heading_deg": np.degrees(desired_heading),
        }

    def get_cross_track_error(self, lat_ac: float, lon_ac: float):
        """Возвращает отклонение от осевой линии в метрах. >0 - правее оси, <0 - левее."""
        d_ac = self.haversine_distance(RWY_START_LAT, RWY_START_LON, lat_ac, lon_ac) / self.R
        theta_ac = self.bearing(RWY_START_LAT, RWY_START_LON, lat_ac, lon_ac)

        xte = np.asin(np.sin(d_ac) * np.sin(theta_ac - self.theta_rwy)) * self.R
        return xte


class VelocityLaw(Enum):
    EQUALLY_SLOW = 1
    GAUSS_BELL = 2


class ReferenceTrajectory:
    """Генератор эталонной кривой скорости (Колокол Гаусса)."""

    def __init__(self, v_start_kts: float, v_target_kts: float, braking_distance_m: float, steering_gain=0.4,
                 law: VelocityLaw = VelocityLaw.GAUSS_BELL):
        self.v_start_ms = v_start_kts * Converts.KTS_TO_MS
        self.v_target_ms = v_target_kts * Converts.KTS_TO_MS
        self.distance = braking_distance_m

        self.steering_gain = steering_gain

        self.law = law

        match law:
            case VelocityLaw.GAUSS_BELL:
                # f(x) = v_start * exp(-x^2 / (2 * b^2))
                # Математический расчет коэффициента 2*b^2 для точного прохождения через точку v_target на дистанции distance
                self.two_b_squared = (self.distance ** 2) / np.log(self.v_start_ms / self.v_target_ms)

            case VelocityLaw.EQUALLY_SLOW:
                # Вычисление требуемого постоянного отрицательного ускорения (модуль)
                # Формула: a = (v_start^2 - v_tgt^2) / (2 * S)
                self.a_req = (self.v_start_ms ** 2 - self.v_target_ms ** 2) / (2.0 * self.distance)

    def get_reference_speed(self, current_distance_m: float) -> float:
        """Возвращает идеальную скорость (м/с) для текущей точки пути."""
        if current_distance_m >= self.distance:
            return self.v_target_ms

        match self.law:
            case VelocityLaw.GAUSS_BELL:
                return self.v_start_ms * np.exp(-(current_distance_m ** 2) / self.two_b_squared)

            case VelocityLaw.EQUALLY_SLOW:
                # v(s) = sqrt(v_0^2 - 2 * a * s)
                val_under_sqrt = self.v_start_ms ** 2 - 2.0 * self.a_req * current_distance_m
                if val_under_sqrt <= 0:
                    return self.v_target_ms

                return np.sqrt(val_under_sqrt)

In [223]:
class PIDController:
    """
    Пропорционально-интегрально-дифференциальный регулятор.
    Улучшенный PID-регулятор с leaky-интегратором и апериодическим фильтром D-составляющей первого порядка.
    """

    def __init__(self, kp: float, ki: float, kd: float, min_out: float = 0.0, max_out: float = 1.0,
                 anti_windup: float = 10.0, integral_decay: float = 0.0, der_filter_tf: float = 0.1, name: str = ""):
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.name = name

        self.min_out = min_out
        self.max_out = max_out

        self.integral = 0.0
        # Avoiding differential shock at startup
        self.prev_error = None

        self.anti_windup = anti_windup
        self.integral_decay = integral_decay  # Коэффициент экспоненциального затухания интеграла
        self.der_filter_tf = der_filter_tf  # Постоянная времени фильтра низких частот D-составляющей (T_f в секундах)
        self.filtered_derivative = 0.0  # Накопленное отфильтрованное значение производной

    def compute(self, error: float, dt: float):
        if dt <= 0.0:
            return 0.0

        if self.integral_decay > 0.0:
            self.integral *= np.exp(-self.integral_decay * dt)

        self.integral += error * dt
        # Anti-windup (защита от накопления интеграла)
        self.integral = max(-self.anti_windup, min(self.anti_windup, self.integral))

        # Экспоненциальный низкочастотный фильтр (Low-Pass Filter / IIR) для производной
        derivative = 0.0
        if self.prev_error is not None:
            raw_derivative = (error - self.prev_error) / dt

            # Расчет весового коэффициента альфа на основе шага dt и постоянной времени фильтра T_f
            # Формула: alpha = dt / (dt + T_f)
            alpha = dt / (dt + self.der_filter_tf)

            # Применение апериодического фильтра первого порядка
            self.filtered_derivative = alpha * raw_derivative + (1.0 - alpha) * self.filtered_derivative
            derivative = self.filtered_derivative
        else:
            self.prev_error = error
            self.filtered_derivative = 0.0

        cprint(f"PID '{self.name}': error={error}, integral={self.integral}, derivative={derivative}", "yellow")

        self.prev_error = error
        output = (self.kp * error) + (self.ki * self.integral) + (self.kd * derivative)

        return max(self.min_out, min(self.max_out, output))

    def reset(self):
        """Сброс внутренних состояний (используется при выключении системы)."""
        self.integral = 0.0
        self.prev_error = None
        self.filtered_derivative = 0.0

In [224]:
class ControllingSystem:
    def __init__(self,
                 xpc: XPlaneConnectX,
                 failure_manager: FailureManager, tracker: RunwayTracker, trajectory: ReferenceTrajectory,
                 runway_center_pid: PIDController,
                 pid_brake_l: PIDController, pid_brake_r: PIDController,
                 pid_rev_l: PIDController, pid_rev_r: PIDController
                 ):
        self.xpc = xpc

        self.failures = failure_manager
        self.tracker = tracker
        self.trajectory = trajectory

        self.current_steering_cmd = 0.0

        self.runway_center_system = RunwayCenteringSystem(xpc, runway_center_pid, tracker)
        self.auto_brake = MultiChannelAutoBrake(xpc, pid_brake_l, pid_brake_r, pid_rev_l, pid_rev_r,
                                                trajectory)

    def control_step(self, dt: float) -> bool:
        rudder_cmd = self.runway_center_system.control(dt)
        cmd_brake_l, cmd_brake_r, cmd_rev_l, cmd_rev_r, break_control = self.auto_brake.control(dt, rudder_cmd)

        if break_control:
            return True

        # Добавляет обработку некорректной конфигурации шасси (имитация неполного контакта стойки с ВПП)
        cmd_brake_l *= self.failures.state.brake_left_eff
        cmd_brake_r *= self.failures.state.brake_right_eff

        # Добавляет обработку нарушения тяги / реверса двигателей
        cmd_rev_l *= self.failures.state.reverse_left_eff
        cmd_rev_l *= self.failures.state.thrust_left_eff
        cmd_rev_r *= self.failures.state.reverse_right_eff
        cmd_rev_r *= self.failures.state.thrust_right_eff

        self.xpc.sendDREF("sim/cockpit2/controls/left_brake_ratio", cmd_brake_l)
        self.xpc.sendDREF("sim/cockpit2/controls/right_brake_ratio", cmd_brake_r)
        self.xpc.sendDREF("sim/cockpit2/engine/actuators/throttle_ratio[0]", cmd_rev_l)
        self.xpc.sendDREF("sim/cockpit2/engine/actuators/throttle_ratio[1]", cmd_rev_r)

        # Добавляем обработку отказа управления носовой стойкой (NWS)
        rudder_cmd *= self.failures.state.steering_eff
        self.xpc.sendDREF("sim/joystick/yoke_heading_ratio", rudder_cmd)

        return False

    def control_exception(self):
        self.auto_brake.control_exception()
        self.runway_center_system.control_exception()


class RunwayCenteringSystem:
    def __init__(self, xpc: XPlaneConnectX, pid: PIDController, tracker: RunwayTracker):
        self.xpc = xpc

        self.pid = pid

        self.tracker = tracker

    def control(self, dt: float) -> float:
        lat = self.xpc.current_dref_values["sim/flightmodel/position/latitude"]['value']
        lon = self.xpc.current_dref_values["sim/flightmodel/position/longitude"]['value']
        heading = self.xpc.current_dref_values["sim/flightmodel2/position/true_psi"]["value"]
        groundspeed_ms = self.xpc.current_dref_values["sim/flightmodel/position/groundspeed"]["value"]

        if None in (lat, lon, heading, groundspeed_ms):
            cprint(f"[RunwayCenteringSystem] Error: drefs values is None", "red")
            return NAN

        guidance = self.tracker.guidance(lat, lon, heading, groundspeed_ms)

        error = guidance["heading_error_deg"]
        rudder_cmd = self.pid.compute(error, dt)

        cprint(
            f"[RunwayCenteringSystem] XTE={guidance['xte']:+6.2f} м | "
            f"Herr={error:+6.2f}° | "
            f"L={guidance['lookahead']:5.1f} м | "
            f"Rudder={rudder_cmd:+.3f}",
            "green"
        )

        return rudder_cmd

    def control_exception(self):
        print("\n[RunwayCenteringSystem] Остановка алгоритма. Возврат управления педалями в ноль.")
        self.xpc.sendDREF("sim/joystick/yoke_heading_ratio", 0.0)


class MultiChannelAutoBrake:
    def __init__(self, xpc: XPlaneConnectX, pid_brake_l: PIDController, pid_brake_r: PIDController,
                 pid_rev_l: PIDController, pid_rev_r: PIDController,
                 trajectory: ReferenceTrajectory):
        self.xpc = xpc

        self.trajectory = trajectory

        self.pid_brake_l = pid_brake_l
        self.pid_brake_r = pid_brake_r
        self.pid_rev_l = pid_rev_l
        self.pid_rev_r = pid_rev_r

        self.traveled_distance_m = 0.0

        print("[MultiChannelAutoBrake] ИСМПУ: Запуск 4-канального торможения. Ожидание...")

    def control(self, dt: float, steering_cmd: float) -> Tuple[float, float, float, float, bool]:
        current_speed_ms = self.xpc.current_dref_values["sim/flightmodel/position/groundspeed"]['value']

        if current_speed_ms is None:
            cprint(f"[MultiChannelAutoBrake] Error: drefs values is None", "red")
            return NAN, NAN, NAN, NAN, True

        self.traveled_distance_m += current_speed_ms * dt
        ref_speed_ms = self.trajectory.get_reference_speed(self.traveled_distance_m)

        current_speed_kts = current_speed_ms * Converts.MS_TO_KTS
        ref_speed_kts = ref_speed_ms * Converts.MS_TO_KTS

        # Глобальная ошибка по скорости. >0 означает, что мы едем слишком быстро
        error = current_speed_ms - ref_speed_ms

        # 1. Расчет тормозов (Hydraulic Brakes)
        cmd_brake_l = self.pid_brake_l.compute(error, dt)
        cmd_brake_r = self.pid_brake_r.compute(error, dt)

        # ---------------------------------------------------------------------
        # Дифференциальное торможение (Steering by braking)
        # Если руль направления неэффективен (малая скорость), рулим тормозами.
        # Отрицательный руль (влево) = нужно зажать левый тормоз сильнее правого.
        # ---------------------------------------------------------------------
        diff_brake = steering_cmd * self.trajectory.steering_gain  # Коэффициент микширования
        cmd_brake_l -= diff_brake
        cmd_brake_r += diff_brake
        # Строгое ограничение гидравлики после микширования
        cmd_brake_l = max(0.0, min(1.0, cmd_brake_l))
        cmd_brake_r = max(0.0, min(1.0, cmd_brake_r))

        # 2. Расчет реверса (Thrust Reversers) с учетом эксплуатационного лимита
        cmd_rev_l = 0.0
        cmd_rev_r = 0.0

        if current_speed_kts > 60.0:
            # Скорость безопасна для реверса
            cmd_rev_l = -self.pid_rev_l.compute(error, dt)  # Инверсия знака для X-Plane
            cmd_rev_r = -self.pid_rev_r.compute(error, dt)
        else:
            # Скорость ниже 60 узлов - принудительное отключение реверса.
            # Сбрасываем интеграторы, чтобы PID не копил ошибку, пока отключен.
            self.pid_rev_l.reset()
            self.pid_rev_r.reset()

        cprint(
            f"[MultiChannelAutoBrake] Dist: {self.traveled_distance_m:4.0f}m | V_cur: {current_speed_kts:3.0f}; V_ref: {ref_speed_kts:3.0f} | "
            f"Brk_L: {cmd_brake_l:.2f}; Brk_R: {cmd_brake_r:.2f} | "
            f"Rev_L: {cmd_rev_l:.2f}; Rev_R: {cmd_rev_r:.2f}", "green")

        if current_speed_ms <= self.trajectory.v_target_ms:
            print("[MultiChannelAutoBrake] Посадочная дистанция пройдена. Скорость руления достигнута.")
            return 0.1, 0.1, 0.0, 0.0, True

        return cmd_brake_l, cmd_brake_r, cmd_rev_l, cmd_rev_r, False

    def control_exception(self):
        print("\n[MultiChannelAutoBrake] Остановка. Сброс всех управляющих поверхностей.")
        self.xpc.sendDREF("sim/cockpit2/controls/left_brake_ratio", 0.0)
        self.xpc.sendDREF("sim/cockpit2/controls/right_brake_ratio", 0.0)
        self.xpc.sendDREF("sim/cockpit2/engine/actuators/throttle_jet_rev_ratio[0]", 0.0)
        self.xpc.sendDREF("sim/cockpit2/engine/actuators/throttle_jet_rev_ratio[1]", 0.0)


In [225]:
# # Коэффициенты подобраны эмпирически. Для тяжелых ЛА (B737) требуется меньше P и больше D.
# # Так как руль направления имеет обратную логику (чтобы повернуть влево (xte < 0), нужно дать отрицательный ввод),
# # знак вывода PID может потребовать инверсии в зависимости от настроек симулятора.
# runway_center_pid = PIDController(kp=0.0006, ki=0.0004, kd=0.045, min_out=-1, max_out=1)
# # runway_center_pid = PIDController(kp=0.0002, ki=0.0007, kd=0.05, min_out=-50, max_out=50)
#
# # Коэффициенты PID подобраны для тяжелого ЛА (масса ~200 тонн)
# # Требуется калибровка под конкретную динамику A330 в X-Plane
# velocity_pid = PIDController(kp=0.15, ki=0.01, kd=0.05, min_out=-50, max_out=50)
#
# # Инициализация 4 независимых контроллеров
# # Коэффициенты для реверса сделаны меньше, так как тяга двигателей
# # имеет огромную инерцию по сравнению с гидравликой тормозов.
# pid_brake_l = PIDController(kp=0.1, ki=0.01, kd=0.05, min_out=-50, max_out=50, name="Brake_L")
# pid_brake_r = PIDController(kp=0.1, ki=0.01, kd=0.05, min_out=-50, max_out=50, name="Brake_R")
#
# # Реверс в X-Plane управляется отрицательным значением throttle_ratio [-1.0, 0.0]
# # Мы настраиваем PID на выдачу от 0 до 1, а инвертировать знак будем перед отправкой.
# # pid_rev_l = PIDController(kp=0.15, ki=0.02, kd=0.0, min_out=0.0, max_out=50, name="Rev_L")
# # pid_rev_r = PIDController(kp=0.15, ki=0.02, kd=0.0, min_out=0.0, max_out=50, name="Rev_R")
# pid_rev_l = PIDController(kp=0.03, ki=0.002, kd=0.01, min_out=0.0, max_out=50, name="Rev_L")
# pid_rev_r = PIDController(kp=0.03, ki=0.002, kd=0.01, min_out=0.0, max_out=50, name="Rev_R")

In [226]:
subscribed_drefs = [
    ("sim/flightmodel/position/latitude", FREQ),
    ("sim/flightmodel/position/longitude", FREQ),
    ("sim/flightmodel/position/groundspeed", FREQ),
    ("sim/flightmodel2/position/true_psi", FREQ),
]

failure_manager = FailureManager()

In [227]:
# #default
# tracker = RunwayTracker(
#     lookahead_min=10.0,
#     lookahead_gain=1.8,
#     xte_gain=2.
# )
# trajectory = ReferenceTrajectory(
#     v_start_kts=INITIAL_SPEED_KTS,
#     v_target_kts=TARGET_SPEED_KTS,
#     braking_distance_m=tracker.haversine_distance(RWY_START_LAT, RWY_START_LON, RWY_END_LAT, RWY_END_LON),
#     steering_gain=0.4
# )
# runway_center_pid = PIDController(kp=0.0015, ki=0.0001, kd=0.065, min_out=-1, max_out=1, integral_decay=0.5, name="Runway_Center")
# pid_brake_l = PIDController(kp=0.1, ki=0.01, kd=0.05, min_out=0.0, max_out=1.0, name="Brake_L")
# pid_brake_r = PIDController(kp=0.1, ki=0.01, kd=0.05, min_out=0.0, max_out=1.0, name="Brake_R")
# pid_rev_l = PIDController(kp=0.03, ki=0.002, kd=0.01, min_out=0.0, max_out=1.0, name="Rev_L")
# pid_rev_r = PIDController(kp=0.03, ki=0.002, kd=0.01, min_out=0.0, max_out=1.0, name="Rev_R")

In [228]:
# # left reverse fail
# tracker = RunwayTracker(
#     lookahead_min=10.0,
#     lookahead_gain=1.6,
#     xte_gain=2.
# )
# trajectory = ReferenceTrajectory(
#     v_start_kts=INITIAL_SPEED_KTS,
#     v_target_kts=TARGET_SPEED_KTS,
#     braking_distance_m=tracker.haversine_distance(RWY_START_LAT, RWY_START_LON, RWY_END_LAT, RWY_END_LON),
#     steering_gain=0.4
# )
#
# failure_manager.activate(FailureMode.REVERSE_LEFT_FAIL)
# pid_brake_l = PIDController(kp=0.1, ki=0.01, kd=0.05, min_out=0.0, max_out=1.0, name="Brake_L") # ignore
#
# runway_center_pid = PIDController(kp=0.0004, ki=0.0006, kd=0.07, min_out=-1, max_out=1, integral_decay=0.15, name="Runway_Center")
# pid_brake_r = PIDController(kp=0.08, ki=0.015, kd=0.06, min_out=0.0, max_out=1.0, name="Brake_R")
# pid_rev_l = PIDController(kp=0.03, ki=0.0025, kd=0.02, min_out=0.0, max_out=1.0, name="Rev_L")
# pid_rev_r = PIDController(kp=0.03, ki=0.0025, kd=0.02, min_out=0.0, max_out=1.0, name="Rev_R")

In [229]:
# # right reverse fail
# tracker = RunwayTracker(
#     lookahead_min=10.0,
#     lookahead_gain=1.6,
#     xte_gain=2.
# )
# trajectory = ReferenceTrajectory(
#     v_start_kts=INITIAL_SPEED_KTS,
#     v_target_kts=TARGET_SPEED_KTS,
#     braking_distance_m=tracker.haversine_distance(RWY_START_LAT, RWY_START_LON, RWY_END_LAT, RWY_END_LON),
#     steering_gain=0.4
# )
#
# failure_manager.activate(FailureMode.REVERSE_RIGHT_FAIL)
# pid_brake_r = PIDController(kp=0.1, ki=0.01, kd=0.05, min_out=0.0, max_out=1.0, name="Brake_L") # ignore
#
# runway_center_pid = PIDController(kp=0.0004, ki=0.0006, kd=0.07, min_out=-1, max_out=1, integral_decay=0.15, name="Runway_Center")
# pid_brake_l = PIDController(kp=0.08, ki=0.015, kd=0.06, min_out=0.0, max_out=1.0, name="Brake_R")
# pid_rev_l = PIDController(kp=0.03, ki=0.0025, kd=0.02, min_out=0.0, max_out=1.0, name="Rev_L")
# pid_rev_r = PIDController(kp=0.03, ki=0.0025, kd=0.02, min_out=0.0, max_out=1.0, name="Rev_R")

In [232]:
# NWS fault
tracker = RunwayTracker(
    lookahead_min=10.0,
    lookahead_gain=1.8,
    xte_gain=2.
)
trajectory = ReferenceTrajectory(
    v_start_kts=INITIAL_SPEED_KTS,
    v_target_kts=TARGET_SPEED_KTS,
    braking_distance_m=tracker.haversine_distance(RWY_START_LAT, RWY_START_LON, RWY_END_LAT, RWY_END_LON),
    steering_gain=1.,
)

failure_manager.activate(FailureMode.NWS_FAIL)
runway_center_pid = PIDController(kp=0.0015, ki=0.0001, kd=0.065, min_out=-1, max_out=1, integral_decay=0.5,
                                  name="Runway_Center")  # ignore

pid_brake_l = PIDController(kp=0.2, ki=0.015, kd=0.065, min_out=0.0, max_out=1.0, integral_decay=0.3, name="Brake_L")
pid_brake_r = PIDController(kp=0.2, ki=0.015, kd=0.065, min_out=0.0, max_out=1.0, integral_decay=0.3, name="Brake_R")
pid_rev_l = PIDController(kp=0.04, ki=0.002, kd=0.01, min_out=0.0, max_out=1.0, name="Rev_L")
pid_rev_r = PIDController(kp=0.04, ki=0.002, kd=0.01, min_out=0.0, max_out=1.0, name="Rev_R")

In [233]:
controller = ControllingSystem(xpc, failure_manager, tracker, trajectory, runway_center_pid, pid_brake_l, pid_brake_r,
                               pid_rev_l, pid_rev_r)

time.sleep(2)
setup_touchdown_uuee(xpc, **MC21_SETUP)

print("Запуск системы удержания оси ВПП...")
xpc.pauseSIM(False)

time.sleep(1.2)
print("Настройка подписки на телеметрию X-Plane...")
xpc.subscribeDREFs(subscribed_drefs, timeout=10.)
last_time = time.time()
try:
    while True:
        current_time = time.time()
        dt = current_time - last_time

        if dt >= DT:
            if controller.control_step(dt):
                raise KeyboardInterrupt

            last_time = current_time

        time.sleep(0.01)  # Снижение нагрузки на CPU

except KeyboardInterrupt:
    controller.control_exception()


[MultiChannelAutoBrake] ИСМПУ: Запуск 4-канального торможения. Ожидание...
Запуск системы удержания оси ВПП...
Настройка подписки на телеметрию X-Plane...
PID 'Runway_Center': error=-0.5192515597451399, integral=-0.02904849636118868, derivative=0.0
[RunwayCenteringSystem] XTE= +0.58 м | Herr= -0.52° | L=191.0 м | Rudder=-0.001
PID 'Brake_L': error=-2.309521887308847, integral=-0.1292016112046057, derivative=0.0
PID 'Brake_R': error=-2.309521887308847, integral=-0.1292016112046057, derivative=0.0
PID 'Rev_L': error=-2.309521887308847, integral=-0.1292016112046057, derivative=0.0
PID 'Rev_R': error=-2.309521887308847, integral=-0.1292016112046057, derivative=0.0
[MultiChannelAutoBrake] Dist:    6m | V_cur: 196; V_ref: 200 | Brk_L: 0.00; Brk_R: 0.00 | Rev_L: -0.00; Rev_R: -0.00
PID 'Runway_Center': error=-0.37418823189609673, integral=-0.04784481085892307, derivative=0.9529004344318431
[RunwayCenteringSystem] XTE= +0.42 м | Herr= -0.37° | L=190.9 м | Rudder=+0.061
PID 'Brake_L': error=-2.